<a href="https://colab.research.google.com/github/JSJeong-me/AI-Innovation-2024/blob/main/303%20gemma_2b_it_sum_ko_qlora_2026_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 2B IT 한국어 뉴스 요약 QLoRA Fine-tuning — Colab (2026)

이 노트북은 기존 `5_3_gemma_2b_it_sum_ko.ipynb`의 목적과 흐름을 유지하면서,
2026년 최신 Hugging Face 패키지 API에 맞게 다시 작성한 Colab 실행용 버전입니다.

**목표**
- Base model: `google/gemma-2b-it`
- Dataset: `daekeun-ml/naver-news-summarization-ko`
- Training: 4-bit QLoRA + TRL `SFTTrainer`
- Output: LoRA adapter + merged standalone model
- Inference: 한국어 뉴스 요약

> **중요:** `google/gemma-2b-it`은 Hugging Face에서 사용 조건 동의 및 로그인이 필요할 수 있습니다.
> Colab에서 **Runtime → Change runtime type → GPU**를 선택하세요.

## 1. 최신 패키지 설치

아래 버전은 2026-09 기준으로 맞춘 실행 조합입니다.

- `transformers==5.16.1`
- `trl==1.13.0`
- `peft==0.20.0`
- `bitsandbytes==0.50.0`
- `accelerate==1.15.0`
- `datasets==5.0.1`

기존 노트북의 `BNB_CUDA_VERSION` 강제 지정 및 별도 Triton 버전 고정은 제거했습니다.

In [1]:
%pip install -q -U \
  "transformers==5.16.1" \
  "trl==1.13.0" \
  "peft==0.20.0" \
  "bitsandbytes==0.50.0" \
  "accelerate==1.15.0" \
  "datasets==5.0.1" \
  "huggingface_hub>=0.34.0" \
  "sentencepiece>=0.2.0" \
  "safetensors>=0.5.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 35.4 MB/s eta 0:00:00


> 이미 오래된 Hugging Face 패키지가 import된 상태에서 설치했다면,
> 이 셀 실행 후 Colab 런타임을 한 번 재시작한 뒤 아래 셀부터 실행하세요.

## 2. 환경 및 버전 확인

In [2]:
import os
import gc
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb
import accelerate

print("Python / PyTorch environment")
print("- torch       :", torch.__version__)
print("- transformers:", transformers.__version__)
print("- datasets    :", datasets.__version__)
print("- peft        :", peft.__version__)
print("- trl         :", trl.__version__)
print("- bitsandbytes:", bnb.__version__)
print("- accelerate  :", accelerate.__version__)
print("- CUDA        :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 활성화되지 않았습니다. Colab Runtime에서 GPU를 선택하세요.")

print("- GPU         :", torch.cuda.get_device_name(0))
print("- BF16 support:", torch.cuda.is_bf16_supported())

Python / PyTorch environment
- torch       : 2.11.0+cu128
- transformers: 5.16.1
- datasets    : 5.0.1
- peft        : 0.20.0
- trl         : 1.13.0
- bitsandbytes: 0.50.0
- accelerate  : 1.15.0
- CUDA        : True
- GPU         : Tesla T4
- BF16 support: True


In [3]:
!nvidia-smi

Thu Sep 10 19:58:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             28W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Hugging Face 로그인

In [9]:
from huggingface_hub import notebook_login
notebook_login()

`google/gemma-2b-it` 모델 페이지에서 라이선스/사용 조건 동의가 필요한 경우 먼저 동의해야 합니다.

## 4. Dataset 로드

In [5]:
from datasets import load_dataset

DATASET_ID = "daekeun-ml/naver-news-summarization-ko"
dataset = load_dataset(DATASET_ID)

print(dataset)

README.md:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 66.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/7.45M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/8.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22194 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2466 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2740 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})


In [6]:
print(dataset["train"][0])

{'date': '2022-07-03 17:14:37', 'category': 'economy', 'press': 'YTN ', 'title': '추경호 중기 수출지원 총력 무역금융 40조 확대', 'document': '앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.', 'link': 'https://n.news.naver.com/mnews/ar

## 5. 공통 설정

In [7]:
BASE_MODEL = "google/gemma-2b-it"

# Colab T4는 FP16, A100/L4 등은 BF16을 사용할 수 있습니다.
USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# 뉴스 원문이 길기 때문에 1024를 기본값으로 사용합니다.
# VRAM 부족 시 512로 낮추세요.
MAX_LENGTH = 1024

print("BASE_MODEL   =", BASE_MODEL)
print("COMPUTE_DTYPE=", COMPUTE_DTYPE)
print("MAX_LENGTH   =", MAX_LENGTH)

BASE_MODEL   = google/gemma-2b-it
COMPUTE_DTYPE= torch.bfloat16
MAX_LENGTH   = 1024


## 6. Base Gemma 모델 요약 추론 (선택 실행)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Gemma는 pad token이 없거나 명시되지 않은 환경에서 eos token을 pad로 사용 가능
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)
baseline_model.eval()

doc = dataset["train"][0]["document"]

messages = [
    {
        "role": "user",
        "content": f"다음 글을 핵심 내용 중심으로 한국어로 요약해주세요.\n\n{doc}",
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(baseline_model.device)

with torch.inference_mode():
    generated = baseline_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = generated[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

**핵심 내용:**

* 앵커 정부는 수출 확대를 위해 총력을 기울이고 있으며, 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다.
* 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다.
* 정부는 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다.


### 6.1 학습 전 GPU 메모리 정리

In [10]:
del baseline_model
gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

NameError: name 'baseline_model' is not defined

## 7. 최신 TRL 형식으로 학습 데이터 변환

최신 `SFTTrainer`는 conversational `prompt` / `completion` 형식을 직접 처리할 수 있습니다.
따라서 기존처럼 `<bos><start_of_turn>...` 문자열을 수동 생성하지 않고,
Gemma tokenizer의 chat template을 사용하도록 구성합니다.

In [ ]:
def to_prompt_completion(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": (
                    "다음 글을 핵심 내용 중심으로 한국어로 요약해주세요.\n\n"
                    + example["document"]
                ),
            }
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["summary"],
            }
        ],
    }

train_data = dataset["train"].map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names,
    desc="Converting train dataset",
)

print(train_data[0])

### 선택: Smoke test

전체 3,000 step 학습 전에 코드를 빠르게 확인하려면 아래처럼 작은 데이터셋을 사용할 수 있습니다.

```python
train_data = train_data.select(range(min(200, len(train_data))))
MAX_STEPS = 20
```

정상 학습에서는 아래 셀의 `MAX_STEPS = 3000`을 사용합니다.

## 8. 4-bit QLoRA 모델 구성

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# QLoRA 학습 준비:
# layer norm / gradient checkpointing 등 k-bit training용 처리
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. TRL SFTTrainer 설정 및 학습

In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "outputs"
ADAPTER_DIR = "gemma-2b-it-sum-ko-lora"

MAX_STEPS = 3000        # 빠른 실행 확인은 20 등으로 낮추세요.
LEARNING_RATE = 2e-4

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="linear",

    # QLoRA/Colab
    optim="paged_adamw_8bit",
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # SFT
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,

    # Logging / saving
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    push_to_hub=False,

    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_data,
    processing_class=tokenizer,
)

trainer.train()

## 10. LoRA Adapter 저장

In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter:", ADAPTER_DIR)

In [ ]:
!du -sh gemma-2b-it-sum-ko-lora
!ls -alh gemma-2b-it-sum-ko-lora | head -30

## 11. LoRA Adapter를 Base Model에 Merge

4-bit quantized training model 자체를 merge하지 않고,
학습 종료 후 메모리를 정리한 뒤 base model을 FP16/BF16으로 다시 로드하여 adapter를 merge합니다.

In [ ]:
# 학습 객체 제거
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

print("Training objects cleared.")

In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

MERGED_DIR = "gemma-2b-it-sum-ko-merged"

# GPU VRAM을 아끼기 위해 CPU에서 merge
base_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="cpu",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

peft_model = PeftModel.from_pretrained(
    base_for_merge,
    ADAPTER_DIR,
)

merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="4GB",
)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved:", MERGED_DIR)

In [ ]:
!du -sh gemma-2b-it-sum-ko-merged
!ls -alh gemma-2b-it-sum-ko-merged | head -30

## 12. Fine-tuned 모델 추론

In [ ]:
# CPU merge 모델 정리 후 GPU inference
del base_for_merge
del peft_model
del merged_model
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer

finetune_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

if finetune_tokenizer.pad_token is None:
    finetune_tokenizer.pad_token = finetune_tokenizer.eos_token

finetune_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)
finetune_model.eval()

In [ ]:
test_doc = dataset["test"][0]["document"]

messages = [
    {
        "role": "user",
        "content": (
            "다음 글을 핵심 내용 중심으로 한국어로 요약해주세요.\n\n"
            + test_doc
        ),
    }
]

inputs = finetune_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(finetune_model.device)

with torch.inference_mode():
    generated = finetune_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=finetune_tokenizer.pad_token_id,
        eos_token_id=finetune_tokenizer.eos_token_id,
    )

new_tokens = generated[0, inputs["input_ids"].shape[1]:]
summary = finetune_tokenizer.decode(new_tokens, skip_special_tokens=True)

print("=== Document ===")
print(test_doc)
print("\n=== Reference Summary ===")
print(dataset["test"][0]["summary"])
print("\n=== Model Summary ===")
print(summary)

## 13. Hugging Face Hub 업로드 (선택)

기존 노트북처럼 shard 파일 하나만 `model.safetensors` 이름으로 올리지 않고,
**폴더 전체를 repository로 업로드**합니다.

아래 `YOUR_HF_USERNAME`을 본인의 Hugging Face 사용자 이름으로 변경하세요.

In [ ]:
from huggingface_hub import HfApi

HF_USERNAME = "YOUR_HF_USERNAME"
ADAPTER_REPO = f"{HF_USERNAME}/gemma-2b-it-sum-ko-lora"
MERGED_REPO = f"{HF_USERNAME}/gemma-2b-it-sum-ko-merged"

api = HfApi()

# LoRA adapter 업로드
api.create_repo(
    repo_id=ADAPTER_REPO,
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=ADAPTER_REPO,
    repo_type="model",
)

print("Uploaded adapter:", ADAPTER_REPO)

# merged model도 필요할 경우 아래 주석을 해제하세요.
# api.create_repo(
#     repo_id=MERGED_REPO,
#     repo_type="model",
#     exist_ok=True,
# )
# api.upload_folder(
#     folder_path=MERGED_DIR,
#     repo_id=MERGED_REPO,
#     repo_type="model",
# )
# print("Uploaded merged model:", MERGED_REPO)

## 14. 주요 변경점

기존 노트북 대비 핵심 수정 사항:

1. 최신 Hugging Face 패키지 버전 고정
2. 별도 Triton 강제 설치 제거
3. `BNB_CUDA_VERSION` 환경 변수 강제 지정 제거
4. `TrainingArguments` 대신 최신 `SFTConfig` 사용
5. `max_seq_length` → `max_length`
6. `warmup_steps=0.03` 오류성 설정 → `warmup_ratio=0.03`
7. 수동 Gemma special-token 문자열 제거
8. conversational `prompt` / `completion` dataset 사용
9. `completion_only_loss=True` 적용
10. `prepare_model_for_kbit_training()` 추가
11. `lora_alpha`, `lora_dropout`, `bias` 명시
12. NF4 + double quantization 사용
13. BF16 지원 GPU에서는 BF16 자동 사용, T4에서는 FP16
14. 학습 후 adapter 저장 및 CPU에서 안전하게 merge
15. Hugging Face Hub에 단일 shard가 아닌 폴더 전체 업로드